In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import random

random.seed(42)

In [0]:
N_CLIENTS = 300
START_DATE = "2023-01-01"
N_MONTHS = 24

In [0]:
# Each row = one client account 
# 1. client_id: unique identifier for each client account
# 2. client_name: name of the client account
# 3. industry: industry of the client account
# 4. region: region of the client account
# 5. company_size: size of the client account
# 6. eligible_employees: number of eligible employees in the client account
# 7. account_age_months_at_start: number of months the client account had been 
# open when the analysis period started (e.g. 0 means they are brand new, 24 means that they had been a client for 2 years)

clients = (
    spark.range(N_CLIENTS)
    .withColumnRenamed("id", "client_id")
    .withColumn("client_name", F.concat(F.lit("Client_"), F.col("client_id")))
    .withColumn(
        "industry",
        F.when(F.rand(seed=1) < 0.25, "Technology")
         .when(F.rand(seed=2) < 0.50, "Healthcare")
         .when(F.rand(seed=3) < 0.75, "Finance")
         .otherwise("Retail")
    )
    .withColumn(
        "region",
        F.when(F.rand(seed=4) < 0.25, "West")
         .when(F.rand(seed=5) < 0.50, "Midwest")
         .when(F.rand(seed=6) < 0.75, "South")
         .otherwise("Northeast")
    )
    .withColumn(
        "company_size",
        F.when(F.rand(seed=7) < 0.35, "Mid-Market")
         .otherwise("Enterprise")
    )
    .withColumn(
        "eligible_employees",
        F.when(F.col("company_size") == "Mid-Market", (F.rand(seed=8) * 3000 + 500).cast("int"))
         .otherwise((F.rand(seed=9) * 20000 + 3000).cast("int"))
    )
    .withColumn(
        "account_age_months_at_start",
        (F.rand(seed=10) * 24).cast("int") 
        #Generate a random number between 0 and 1 and multiply it by 24 to get a random number between 0 and 24
    )
)

display(clients)

In [0]:
display(spark.table("clients"))

In [0]:
calendar = (
    spark.range(N_MONTHS)
    .withColumnRenamed("id","month_index")
    .withColumn("month", F.add_months(F.to_date(F.lit(START_DATE)), F.col("month_index")))
)

display(calendar)

In [0]:
contracts = (
    clients.withColumn("contract_start_date", F.add_months(F.to_date(F.lit(START_DATE)), (F.rand(seed=11)*6).cast("int")))
    .withColumn("contract_term_months", F.when(F.rand(seed=12) < 0.75, 12).otherwise(24))
    .withColumn("contract_end_date", F.add_months(F.col("contract_start_date"), F.col("contract_term_months")))
    .withColumn("annual_contract_value", (F.col("eligible_employees") * F.when(F.col("company_size") == "Mid-Market", F.lit(18)).otherwise(F.lit(24)) * 12).cast("int"))
    .select("client_id"
            ,"client_name"
            ,"industry"
            ,"region"
            ,"company_size"
            ,"eligible_employees"
            ,"contract_start_date"
            ,"contract_term_months"
            ,"contract_end_date"
            ,"annual_contract_value")
)

display(contracts)

In [0]:
monthly_base = (
    contracts.crossJoin(calendar)
    .filter(F.col("month") >= F.date_trunc("month", F.col("contract_start_date")))
    .filter(F.col("month") <= F.date_trunc("month", F.col("contract_end_date")))
)

display(monthly_base)

# Baseline Structural Risk
Introducing Weak Clients

In [0]:
# This table create a baseline structural risk, introducing weak clients that will be weak consistenty across time (not declining towards renewal)
client_health = (
    clients
    .select("client_id")
    .withColumn(
        "latent_risk_segment",
        F.when(F.rand(seed=18) < 0.15, "high_risk")
         .when(F.rand(seed=19) < 0.35, "medium_risk")
         .otherwise("low_risk")
    )
)

display(client_health)

In [0]:
monthly_engagement = (
    monthly_base
    .join(client_health, on="client_id", how="left")
    .withColumn(
        "risk_multiplier",
        F.when(F.col("latent_risk_segment") == "high_risk", 0.65)
         .when(F.col("latent_risk_segment") == "medium_risk", 0.85)
         .otherwise(1.0)
    )
    .withColumn(
        "enrolled_members",
        (
            F.col("eligible_employees") *
            (
                F.when(F.col("company_size") == "Mid-Market", 0.10).otherwise(0.14)
                + F.rand(seed=13) * 0.08 
            ) * F.col("risk_multiplier")
        ).cast("int")
    )
    .withColumn(
        "active_members",
        (
            F.col("enrolled_members") *
            (0.45 + F.rand(seed=14) * 0.35)
        ).cast("int")
    )
    .withColumn(
        "sessions",
        (
            F.col("active_members") *
            (1.2 + F.rand(seed=15) * 1.8)
        ).cast("int")
    )
    .withColumn(
        "webinar_attendance",
        (
            F.col("eligible_employees") *
            (0.005 + F.rand(seed=16) * 0.02) *
            F.col("risk_multiplier")
        ).cast("int")
    )
    .withColumn(
        "employer_comms_sent",
        (F.rand(seed=17) * 6).cast("int")
    )
    .withColumn(
        "utilization_rate",
        F.round(F.col("active_members") / F.col("eligible_employees"), 4)
    )
    .select(
        "client_id",
        "month",
        "enrolled_members",
        "active_members",
        "sessions",
        "webinar_attendance",
        "employer_comms_sent",
        "utilization_rate"
    )
)

display(monthly_engagement)

In [0]:
# Risky clients often have more support pain
support_tickets = (
    monthly_base
    .join(client_health, on="client_id", how="left")
    .withColumn(
        "ticket_count",
        F.when(F.col("latent_risk_segment") == "high_risk", (F.rand(seed=20) * 8 + 3).cast("int"))
         .when(F.col("latent_risk_segment") == "medium_risk", (F.rand(seed=21) * 5 + 1).cast("int"))
         .otherwise((F.rand(seed=22) * 3).cast("int"))
    )
    .withColumn(
        "high_severity_tickets",
        F.when(F.col("latent_risk_segment") == "high_risk", (F.rand(seed=23) * 3).cast("int"))
         .otherwise((F.rand(seed=24) * 2).cast("int"))
    )
    .select(
        "client_id",
        "month",
        "ticket_count",
        "high_severity_tickets"
    )
)

display(support_tickets)

In [0]:
# We want healthy clients to have better satisfaction
nps_scores = (monthly_base
              .join(client_health, on="client_id", how="left")
              .withColumn(
                  "nps",
                  F.when(F.col("latent_risk_segment") == "high_risk", (F.rand(seed=25) * 30 + 10).cast("int"))
                   .when(F.col("latent_risk_segment") == "medium_risk", (F.rand(seed=26) * 25 - 30).cast("int"))
                   .otherwise((F.rand(seed=27) * 20 + 50).cast("int"))
              )
              .select(
                  "client_id"
                  , "month"
                  ,"nps"              )
             )

display(nps_scores)

In [0]:
renewals = (
    contracts
    .join(client_health, on="client_id", how="left")
    .withColumn(
        "renewed_flag",
        F.when(F.col("latent_risk_segment") == "high_risk",
               F.when(F.rand(seed=28) < 0.35, 1).otherwise(0))
         .when(F.col("latent_risk_segment") == "medium_risk",
               F.when(F.rand(seed=29) < 0.70, 1).otherwise(0))
         .otherwise(
               F.when(F.rand(seed=30) < 0.92, 1).otherwise(0)
         )
    )
    .withColumn("non_renewal_flag", 1 - F.col("renewed_flag"))
    .withColumn(
        "renewal_cycle_id",
        F.concat(F.lit("R_"), F.col("client_id"))
    )
    .select(
        "client_id",
        "renewal_cycle_id",
        "contract_start_date",
        "contract_end_date",
        "annual_contract_value",
        "renewed_flag",
        "non_renewal_flag"
    )
)

display(renewals)

In [0]:
clients.write.mode("overwrite").saveAsTable("clients")
contracts.write.mode("overwrite").saveAsTable("contracts")
monthly_engagement.write.mode("overwrite").saveAsTable("monthly_engagement")
support_tickets.write.mode("overwrite").saveAsTable("support_tickets")
nps_scores.write.mode("overwrite").saveAsTable("nps_scores")
renewals.write.mode("overwrite").saveAsTable("renewals")
client_health.write.mode("overwrite").saveAsTable("client_health")

## Sanity Checks

In [0]:
spark.table("clients").count()

In [0]:
display(
    spark.table("renewals")
    .groupBy("renewed_flag")
    .count()
)

In [0]:
display(
    spark.table("monthly_engagement")
    .select(F.avg("utilization_rate").alias("avg_utilization"))
)

# Add time-based deterioration near renewal
To make the synthetic dataset more realistic, I introduced time-based deterioration in the final six months before contract renewal. Higher-risk clients exhibit declining utilization and engagement, rising support burden, and falling NPS as contract end approaches. This creates realistic temporal signals for modeling renewal risk rather than relying only on static client differences.

In [0]:
from pyspark.sql import functions as F

monthly_engagement = (
    monthly_base
    .join(client_health, on="client_id", how="left")
    
    # months remaining until renewal
    .withColumn(
        "months_to_renewal",
        F.floor(F.months_between(F.col("contract_end_date"), F.col("month")))
    )
    
    # baseline risk multiplier from latent segment
    .withColumn(
        "baseline_risk_multiplier",
        F.when(F.col("latent_risk_segment") == "high_risk", 0.65)
         .when(F.col("latent_risk_segment") == "medium_risk", 0.85)
         .otherwise(1.0)
    )
    
    # deterioration only in final 6 months before renewal. Acts as a countdown stress variable
    .withColumn(
        "renewal_pressure",
        F.when(F.col("months_to_renewal") <= 6, (6 - F.col("months_to_renewal")) / 6.0) 
        # The closer to renewal (smaller months_to_renewal) the higher the pressure.
        # One month before renewal the pressure is 0.83
        # Five months before renewal the pressure is 0.16
         .otherwise(0.0)
    )
    
    # stronger deterioration for riskier clients
    .withColumn(
        "deterioration_multiplier",
        F.when(
            F.col("latent_risk_segment") == "high_risk",
            1.0 - 0.35 * F.col("renewal_pressure")
            # Max drop in engagement 35%. Engagement would be at least 65% in the worst case scenario
        ).when(
            F.col("latent_risk_segment") == "medium_risk",
            1.0 - 0.18 * F.col("renewal_pressure")
            # Max drop in engagement 18%. Engagement would be at least 82%
        ).otherwise(1.0 - 0.05 * F.col("renewal_pressure"))
        # Max drop in engagement 5%. Engagement would be at least 95%
        # 
    )
    
    # enrolled members
    .withColumn(
        "enrolled_members",
        (
            F.col("eligible_employees") *
            (
                F.when(F.col("company_size") == "Mid-Market", 0.10).otherwise(0.14)
                + F.rand(seed=13) * 0.08 #Adding random variation between 0% and 8% (Mid-Market can have between 10% and 18% of eligible_employees. While Enterprise can have something between 14% and 22% of eligible_employees)
            ) *
            F.col("baseline_risk_multiplier") *
            F.col("deterioration_multiplier")
        ).cast("int")
    )
    
    # active members
    .withColumn(
        "active_members",
        (
            F.col("enrolled_members") *
            (0.45 + F.rand(seed=14) * 0.35) * 
            #Create an activity rate between 45% and 80% of enrolled_members that are active in a given month. 0.45 -> 45% = minimum activity rate, ensures that even low engagement months still have some baseline activity
            F.col("deterioration_multiplier")
            # Baseline_risk_multiplier not applied because it was already applied in the calculation of eligible_employees, doing it again would double-penalize risky clients.

        ).cast("int")
    )
    
    # sessions
    .withColumn(
        "sessions",
        (
            F.col("active_members") *
            (1.2 + F.rand(seed=15) * 1.8) *
            F.col("deterioration_multiplier")
        ).cast("int")
    )
    
    # webinar attendance
    .withColumn(
        "webinar_attendance",
        (
            F.col("eligible_employees") *
            (0.005 + F.rand(seed=16) * 0.02) *
            F.col("baseline_risk_multiplier") *
            F.col("deterioration_multiplier")
        ).cast("int")
    )
    
    # employer comms: mostly stable, mild decline for risky accounts
    .withColumn(
        "employer_comms_sent",
        (
            (F.rand(seed=17) * 6) * 
            # Generates a randome number between 0 and 6 representing the number of communications sent in a month
            (1.0 - 0.10 * F.col("renewal_pressure"))
            # Communication volume can decline up to 10% near renewal. Only 10% because HR behavior doesn't usually colapse they way engagement metrics do (because the may be automated or campaigns are often schedule some time in advance)
            # Risk multipliers are not applied to comms to let them vary independently from engagement metrics. low engagement + high comms = HR trying to fix adoption. low engagement + low comms = benefit being ignored internally
        ).cast("int")
    )
    
    .withColumn(
        "utilization_rate",
        F.round(F.col("active_members") / F.col("eligible_employees"), 4)
    )
    
    .select(
        "client_id",
        "month",
        "months_to_renewal",
        "latent_risk_segment",
        "enrolled_members",
        "active_members",
        "sessions",
        "webinar_attendance",
        "employer_comms_sent",
        "utilization_rate"
    )
)

display(monthly_engagement)

In [0]:
support_tickets = (
    monthly_base
    .join(client_health, on="client_id", how="left")
    .withColumn(
        "months_to_renewal",
        F.floor(F.months_between(F.col("contract_end_date"), F.col("month")))
    )
    .withColumn(
        "renewal_pressure",
        F.when(F.col("months_to_renewal") <= 6, (6 - F.col("months_to_renewal")) / 6.0)
         .otherwise(0.0)
    )
    .withColumn(
        "base_ticket_count",
        F.when(F.col("latent_risk_segment") == "high_risk", (F.rand(seed=20) * 8 + 3))
         .when(F.col("latent_risk_segment") == "medium_risk", (F.rand(seed=21) * 5 + 1))
         .otherwise((F.rand(seed=22) * 3 + 0.5))
    )
    .withColumn(
        "ticket_escalation_multiplier",
        F.when(F.col("latent_risk_segment") == "high_risk", 1.0 + 0.80 * F.col("renewal_pressure"))
         .when(F.col("latent_risk_segment") == "medium_risk", 1.0 + 0.40 * F.col("renewal_pressure"))
         .otherwise(1.0 + 0.10 * F.col("renewal_pressure"))
    )
    .withColumn(
        "ticket_count",
        (F.col("base_ticket_count") * F.col("ticket_escalation_multiplier")).cast("int")
    )
    .withColumn(
        "high_severity_tickets",
        F.when(
            F.col("latent_risk_segment") == "high_risk",
            (F.rand(seed=23) * 3 * F.col("ticket_escalation_multiplier")).cast("int")
        ).otherwise(
            (F.rand(seed=24) * 2 * F.col("ticket_escalation_multiplier")).cast("int")
        )
    )
    .select(
        "client_id",
        "month",
        "months_to_renewal",
        "ticket_count",
        "high_severity_tickets"
    )
)

display(support_tickets)

In [0]:
nps_scores = (
    monthly_base
    .join(client_health, on="client_id", how="left")
    .withColumn(
        "months_to_renewal",
        F.floor(F.months_between(F.col("contract_end_date"), F.col("month")))
    )
    .withColumn(
        "renewal_pressure",
        F.when(F.col("months_to_renewal") <= 6, (6 - F.col("months_to_renewal")) / 6.0)
         .otherwise(0.0)
    )
    .withColumn(
        "base_nps",
        F.when(F.col("latent_risk_segment") == "high_risk", (F.rand(seed=25) * 30 + 10))
         .when(F.col("latent_risk_segment") == "medium_risk", (F.rand(seed=26) * 25 + 30))
         .otherwise((F.rand(seed=27) * 20 + 50))
    )
    .withColumn(
        "nps_decline",
        F.when(F.col("latent_risk_segment") == "high_risk", 18 * F.col("renewal_pressure"))
         .when(F.col("latent_risk_segment") == "medium_risk", 10 * F.col("renewal_pressure"))
         .otherwise(3 * F.col("renewal_pressure"))
    )
    .withColumn(
        "nps",
        F.round(F.col("base_nps") - F.col("nps_decline")).cast("int")
    )
    .withColumn(
        "nps",
        F.when(F.col("nps") < 0, 0).otherwise(F.col("nps"))
    )
    .select(
        "client_id",
        "month",
        "months_to_renewal",
        "nps"
    )
)

display(nps_scores)

In [0]:
monthly_engagement.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("monthly_engagement")
support_tickets.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("support_tickets")
nps_scores.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("nps_scores")

## Deterioration Sanity Checks

In [0]:
high_risk_client = (
    client_health
    .filter(F.col("latent_risk_segment") == "high_risk")
    .limit(1)
    .collect()[0]["client_id"]
)

print("High-risk client_id:", high_risk_client)

In [0]:
display(
    spark.table("monthly_engagement")
    .filter(F.col("client_id") == high_risk_client)
    .orderBy("month")
)

In [0]:
display(
    spark.table("support_tickets")
    .filter(F.col("client_id") == high_risk_client)
    .orderBy("month")
)

In [0]:
display(
    spark.table("nps_scores")
    .filter(F.col("client_id") == high_risk_client)
    .orderBy("month")
)

In [0]:
# Compare average patterns by risk segment 
display(
    monthly_engagement
    .groupBy("latent_risk_segment", "months_to_renewal")
    .agg(
        F.avg("utilization_rate").alias("avg_utilization_rate"),
        F.avg("sessions").alias("avg_sessions")
    )
    .orderBy("latent_risk_segment", "months_to_renewal")
)

In [0]:
display(
    support_tickets
    .join(client_health, on="client_id", how="left")
    .groupBy("latent_risk_segment", "months_to_renewal")
    .agg(F.avg("ticket_count").alias("avg_ticket_count"))
    .orderBy("latent_risk_segment", "months_to_renewal")
)

In [0]:
display(
    nps_scores
    .join(client_health, on="client_id", how="left")
    .groupBy("latent_risk_segment", "months_to_renewal")
    .agg(F.avg("nps").alias("avg_nps"))
    .orderBy("latent_risk_segment", "months_to_renewal")
)